In [10]:
import numpy as np
from pathlib import Path
from scipy.interpolate import RegularGridInterpolator
from astropy import units as u
from astropy import constants as const
import matplotlib.pyplot as plt
import pandas as pd
import os
import re

In [16]:
def load_koester_model(
    filename=None,
    teff=None,
    logg=None,
    path="/Users/leighharwell2/Desktop/DW/IRchara/Model/models_1792598326/koester2",  # <-- Added your exact folder path here
):
    """Loads a Koester white dwarf model file from its subfolder path."""
    if filename is None:
        if teff is None or logg is None:
            raise ValueError(
                "You must provide either a 'filename' or both 'teff' and 'logg'."
            )

        teff_str = f"{int(teff):05d}"
        logg_str = f"{int(round(logg * 100)):03d}"
        filename = f"da{teff_str}_{logg_str}.dk.dat.txt"

    # Combine the folder path and the filename safely
    full_path = os.path.join(path, filename)

    header_teff = None
    header_logg = None

    # Parse the header for Teff and log g sanity checks
    with open(full_path, "r") as f:
        for line in f:
            if not line.startswith("#"):
                break

            teff_match = re.search(r"teff\s*=\s*(\d+)", line, re.IGNORECASE)
            if teff_match:
                header_teff = int(teff_match.group(1))

            logg_match = re.search(
                r"logg\s*=\s*([\d.]+)", line, re.IGNORECASE
            )
            if logg_match:
                header_logg = float(logg_match.group(1))

    # Sanity checks
    if teff is not None and header_teff is not None:
        assert (
            header_teff == teff
        ), f"Sanity check failed: Expected Teff {teff}, found {header_teff}"
    if logg is not None and header_logg is not None:
        assert (
            np.isclose(header_logg, logg)
        ), f"Sanity check failed: Expected log g {logg}, found {header_logg}"

    print(f"--- Successfully parsed {filename} ---")
    if header_teff and header_logg:
        print(f"Header Metadata -> Teff: {header_teff} K, log g: {header_logg}")

    # Load the data columns
    wavelength, flux = np.loadtxt(full_path, comments="#", unpack=True)

    return wavelength, flux


# ==========================================
# Execution Test
# ==========================================
# This will now correctly look inside "models_1792598326/koester2/da05000_650.dk.dat.txt"
wl1, flux1 = load_koester_model(teff=5000, logg=6.5)

print(f"Number of data points: {len(wl1)}")
print(f"Wavelength range: {wl1[0]:.2f} Å to {wl1[-1]:.2f} Å\n")


--- Successfully parsed da05000_650.dk.dat.txt ---
Header Metadata -> Teff: 5000 K, log g: 6.5
Number of data points: 3167
Wavelength range: 899.23 Å to 29991.79 Å



In [17]:
# ==========================================
# PART 1: Read one file, print stats, and plot
# ==========================================
# Let's say your first file is Teff=5000, logg=6.5
wl1, flux1 = load_koester_model(teff=5000, logg=6.5)

print(f"Number of data points: {len(wl1)}")
print(f"Wavelength range: {wl1[0]:.2f} Å to {wl1[-1]:.2f} Å\n")


--- Successfully parsed da05000_650.dk.dat.txt ---
Header Metadata -> Teff: 5000 K, log g: 6.5
Number of data points: 3167
Wavelength range: 899.23 Å to 29991.79 Å

